# 生成百万级带异常注入的 CSV 测试数据

本脚本用于生成 `large_data.csv` 文件，包含 100 万行测试数据，并注入各类"杂质"用于测试数据处理代码的健壮性。

## 异常注入类型
- 空 event_id（0.3%）
- 默认 user_id（-1 或 guest，1.5%）
- Unix 毫秒时间戳（5%）
- event_type 拼写错误（1.5%）
- 特殊字符 metadata（10%）

In [ ]:
import csv
import json
import random
import uuid
from datetime import datetime, timedelta

In [ ]:
# ==================== 配置参数 ====================
TOTAL_ROWS = 1_000_000  # 总行数
OUTPUT_FILE = "large_data.csv"

# 异常注入比例配置
EMPTY_EVENT_ID_RATIO = 0.003    # 0.3% 空 event_id
DEFAULT_USER_ID_RATIO = 0.015   # 1.5% 默认 user_id
UNIX_TIMESTAMP_RATIO = 0.05     # 5% Unix 毫秒时间戳
TYPO_EVENT_TYPE_RATIO = 0.015   # 1.5% 拼写错误

# 正常的事件类型
NORMAL_EVENT_TYPES = ["login", "logout", "click", "view", "purchase", "search"]

# 拼写错误映射
TYPO_EVENT_TYPES = {
    "click": ["clic", "clik", "cllick", "ckick"],
    "login": ["logn", "login ", "LogIn", "loign"],
    "view": ["vew", "viwe", "vieew"],
    "logout": ["logut", "logot", "log-out"],
    "purchase": ["puchase", "purhcase", "purchse"],
    "search": ["serch", "seach", "serach"]
}

# 设备信息选项
OS_OPTIONS = ["Windows", "macOS", "Linux", "Android", "iOS"]
BROWSER_OPTIONS = ["Chrome", "Firefox", "Safari", "Edge"]
SCREEN_OPTIONS = ["1920x1080", "1366x768", "2560x1440", "3840x2160", "1440x900", "750x1334"]

# 特殊字符测试文本
SPECIAL_CHARS_LIST = [
    "包含，逗号",
    '包含"引号"',
    "包含\n换行符",
    "包含\r\nWindows 换行",
    "特殊字符：@#$%^&*()",
    "Unicode 测试：中文 日本語 한국어",
    "Emoji 测试：😀😁😂",
    "混合测试，带\"引号\"和\n换行",
    "制表符\t测试",
    "反斜杠\\测试"
]

NORMAL_METADATA = [
    "正常用户行为记录",
    "页面浏览统计",
    "用户交互事件",
    "系统自动记录",
    "API 调用日志",
    "前端埋点数据",
    "后端服务日志",
    "用户操作审计"
]

print("配置加载完成")

In [ ]:
# ==================== 数据生成函数 ====================

def generate_event_id(is_empty=False):
    """生成 event_id"""
    if is_empty:
        return ""
    return f"evt_{uuid.uuid4()}"


def generate_user_id(is_default=False):
    """生成 user_id"""
    if is_default:
        return random.choice(["-1", "guest"])
    if random.random() < 0.5:
        return f"user_{random.randint(1000, 99999)}"
    else:
        return str(random.randint(10000, 999999))


def generate_action_time(use_unix=False):
    """生成 action_time"""
    base_time = datetime(2023, 1, 1)
    random_days = timedelta(days=random.randint(0, 1095))
    random_seconds = timedelta(seconds=random.randint(0, 86399))
    event_time = base_time + random_days + random_seconds

    if use_unix:
        return str(int(event_time.timestamp() * 1000))
    else:
        return event_time.strftime("%Y-%m-%dT%H:%M:%SZ")


def generate_event_type(has_typo=False):
    """生成 event_type"""
    normal_type = random.choice(NORMAL_EVENT_TYPES)
    if has_typo:
        return random.choice(TYPO_EVENT_TYPES.get(normal_type, [normal_type]))
    return normal_type


def generate_device_info():
    """生成 device_info JSON 字符串"""
    device = {
        "os": random.choice(OS_OPTIONS),
        "browser": random.choice(BROWSER_OPTIONS),
        "version": f"{random.randint(80, 130)}.{random.randint(0, 9)}.{random.randint(0, 99)}",
        "screen": random.choice(SCREEN_OPTIONS),
        "language": random.choice(["zh-CN", "en-US", "ja-JP", "ko-KR"]),
        "timezone": random.choice(["UTC+8", "UTC+0", "UTC+9", "UTC-5"])
    }
    return json.dumps(device, ensure_ascii=False)


def generate_metadata():
    """生成 metadata，包含特殊字符"""
    if random.random() < 0.1:
        base = random.choice(SPECIAL_CHARS_LIST)
    else:
        base = random.choice(NORMAL_METADATA)
    extra = f" [seq:{random.randint(1, 100000)}]"
    return base + extra


def generate_row():
    """生成单行数据"""
    is_empty_event = random.random() < EMPTY_EVENT_ID_RATIO
    is_default_user = random.random() < DEFAULT_USER_ID_RATIO
    is_unix_time = random.random() < UNIX_TIMESTAMP_RATIO
    has_typo = random.random() < TYPO_EVENT_TYPE_RATIO

    row = {
        "event_id": generate_event_id(is_empty_event),
        "user_id": generate_user_id(is_default_user),
        "action_time": generate_action_time(is_unix_time),
        "event_type": generate_event_type(has_typo),
        "device_info": generate_device_info(),
        "metadata": generate_metadata()
    }

    return row, {
        "empty_event": is_empty_event,
        "default_user": is_default_user,
        "unix_time": is_unix_time,
        "typo": has_typo
    }

print("数据生成函数定义完成")

In [ ]:
# ==================== 主程序 ====================
import time

print(f"开始生成 {TOTAL_ROWS:,} 行测试数据...")
start_time = time.time()

# 统计信息
stats = {
    "total": 0,
    "empty_event_id": 0,
    "default_user_id": 0,
    "unix_timestamp": 0,
    "typo_event_type": 0,
    "special_metadata": 0
}

# 使用流式写入，避免内存溢出
batch_size = 100000

with open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as f:
    fieldnames = ["event_id", "user_id", "action_time", "event_type", "device_info", "metadata"]
    writer = csv.DictWriter(f, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)

    # 写入表头
    writer.writeheader()

    for i in range(TOTAL_ROWS):
        row, flags = generate_row()
        writer.writerow(row)

        # 更新统计
        stats["total"] += 1
        if flags["empty_event"]:
            stats["empty_event_id"] += 1
        if flags["default_user"]:
            stats["default_user_id"] += 1
        if flags["unix_time"]:
            stats["unix_timestamp"] += 1
        if flags["typo"]:
            stats["typo_event_type"] += 1
        if any(sc in row["metadata"] for sc in SPECIAL_CHARS_LIST):
            stats["special_metadata"] += 1

        # 进度报告
        if (i + 1) % batch_size == 0:
            elapsed = time.time() - start_time
            print(f"已生成 {i + 1:,} 行 (耗时：{elapsed:.1f}秒)...")

end_time = time.time()
total_time = end_time - start_time

# 输出统计报告
print("\n" + "=" * 60)
print("数据生成完成！统计信息如下：")
print("=" * 60)
print(f"总行数：{stats['total']:,}")
print(f"生成耗时：{total_time:.2f} 秒 ({total_time/60:.2f} 分钟)")
print(f"空 event_id 数量：{stats['empty_event_id']:,} ({stats['empty_event_id']/stats['total']*100:.3f}%)")
print(f"默认 user_id 数量：{stats['default_user_id']:,} ({stats['default_user_id']/stats['total']*100:.3f}%)")
print(f"Unix 毫秒时间戳数量：{stats['unix_timestamp']:,} ({stats['unix_timestamp']/stats['total']*100:.3f}%)")
print(f"拼写错误 event_type 数量：{stats['typo_event_type']:,} ({stats['typo_event_type']/stats['total']*100:.3f}%)")
print(f"包含特殊字符的 metadata 数量：{stats['special_metadata']:,}")
print("=" * 60)
print(f"文件已保存至：{OUTPUT_FILE}")

## 数据验证

生成完成后，可以简单验证数据：

In [ ]:
# 验证生成的文件
import os
import pandas as pd

file_size = os.path.getsize(OUTPUT_FILE)
print(f"文件大小：{file_size / (1024*1024):.2f} MB")

# 读取前 5 行预览
df_sample = pd.read_csv(OUTPUT_FILE, nrows=5)
print("\n数据预览（前 5 行）：")
df_sample